In [1]:
from pathlib import Path

MODEL_ID = "google/gemma-4-E2B-it"
WEIGHTS  = Path("model_weights")

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Run our PyTorch-side tokenizer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.tokenization import GemmaTokenizer

tok = GemmaTokenizer(WEIGHTS / "tokenizer.json")

s = "Explain MoE in transformers in 3 sentences."
ids = tok.encode(s)

print(f"vocab_size = {tok.vocab_size:,}")
print(f"ids ({len(ids)}): {ids[:10]} ...")

for tid, piece in tok.pretty(ids)[:10]:
    print(f"  {tid:>6}  {piece!r}")

print(f"\nroundtrip: {tok.decode(ids)!r}")

vocab_size = 262,144
ids (10): [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761] ...
  155122  'Explain'
    7945  '▁Mo'
  236788  'E'
     528  '▁in'
   39687  '▁transformers'
     528  '▁in'
  236743  '▁'
  236800  '3'
   23974  '▁sentences'
  236761  '.'

roundtrip: 'Explain MoE in transformers in 3 sentences.'


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embed via OUR GemmaEmbedding  (pure nn.Embedding lookup)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding import GemmaEmbedding

emb  = GemmaEmbedding.from_safetensors(WEIGHTS / "model.safetensors")
ours = emb(torch.tensor(ids, dtype=torch.long))

print(f"weight : {tuple(emb.embed.weight.shape)}  dtype={emb.embed.weight.dtype}")
print(f"vecs   : {tuple(ours.shape)}")
print(f"ours[0, :6] = {ours[0, :6].tolist()}")

weight : (262144, 1536)  dtype=torch.bfloat16
vecs   : (10, 1536)
ours[0, :6] = [-0.1123046875, 0.1826171875, -0.08251953125, -0.7265625, 2.171875, -1.484375]
